# Building a Local RAG System — Tasks 2.1 to 2.7
Executable assignment walkthrough: Unstructured hi-res parsing, LangChain chunking, BGE embeddings, Weaviate, retrieval, reranking, and Qwen generation.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from rag.exact_pipeline import *
print('Project:', ROOT)
print('PDF:', PDF, PDF.exists())

## 2.1 Parse with Unstructured `hi_res`
This layout-aware strategy identifies titles, narrative text, lists, tables, and images while retaining page metadata.

In [ ]:
elements = parse_hi_res()
parse_stats = element_report(elements)
display(parse_stats)
for element in elements[:5]:
    print(type(element).__name__, getattr(element.metadata, 'page_number', None), str(element)[:160])

## Cleaning
Cleaning is intentionally conservative: collapse repeated whitespace while preserving control numbers, safeguard identifiers, punctuation, terminology, and page metadata.

In [ ]:
documents = elements_to_documents(elements)
print('Clean elements:', len(documents))
display(documents[0])

## 2.2 Compare chunking strategies
We compare recursive character, token-aware, and layout-aware `by_title` chunking, then select `by_title` for ingestion.

In [ ]:
chunks = chunk_comparison(elements)
display(chunk_report(chunks))
selected_chunks = chunks['by_title']
print(selected_chunks[0].page_content[:500], selected_chunks[0].metadata)

## 2.3 Embed with BGE-small
The requested LangChain Hugging Face wrapper loads `BAAI/bge-small-en-v1.5` locally. Each normalized vector has 384 dimensions.

In [ ]:
embeddings = langchain_embeddings()
sample_vector = embeddings.embed_query('How often should enterprise assets be inventoried?')
print('Dimensions:', len(sample_vector), 'First values:', sample_vector[:5])

## 2.4–2.5 Store and ingest in Weaviate
Start the local service first with `docker compose up -d`. The collection stores text, vectors, source, page, category, and element identifiers.

In [ ]:
client, store = store_in_weaviate(selected_chunks)
try:
    print('Objects:', client.collections.get(COLLECTION).aggregate.over_all(total_count=True).total_count)
    display(verify_weaviate(store))
finally:
    client.close()

## 2.6–2.7 Retrieve, rerank, and generate
The live Sentinel engine combines BM25 and BGE retrieval, reranks ambiguous candidates with MiniLM, and sends focused evidence to local Qwen3:4b through Ollama.

In [ ]:
from rag.engine import RAGEngine
engine = RAGEngine()
question = 'An employee left the company but can still log in. What should we do?'
result = engine.answer(question)
print(result['mode'], result['answer'])
display([{k: s[k] for k in ('source','page','score')} for s in result['sources']])

## 3.1–3.2 Evaluation and HITL
Run `python evaluation/evaluate.py` for deterministic retrieval metrics, `python evaluation/compare.py` for the BM25/BGE/reranker ablation, and `python evaluation/deepeval_run.py` for the five DeepEval RAG metrics plus safe abstention. Feedback from the frontend is stored in `.rag_data/feedback.jsonl`.